# take a look at A Tool

In [2]:
from langchain_core.tools import tool
from llms.langchain_tutorial.deepseek import get_deepseek_chat

@tool
def count_characters(text: str) -> int:
    """计算文本中的字符个数。"""
    return len(text)

tool = count_characters
{'name': tool.name, 'description': tool.description, 'input': tool.args}

{'name': 'count_characters',
 'description': 'count_characters(text: str) -> int - 计算文本中的字符个数。',
 'input': {'text': {'title': 'Text', 'type': 'string'}}}

In [ ]:
from langchain_core.messages import HumanMessage, ToolMessage

model_with_tool = get_deepseek_chat().bind_tools([tool])
messages = [HumanMessage(content='请用工具计算“langchain”有几个字符。')]
answer = model_with_tool.invoke(messages)
if answer.tool_calls:
    messages.append(answer)
    for call in answer.tool_calls:
        result = tool.invoke(call['args'])
        messages.append(ToolMessage(content=str(result), tool_call_id=call['id']))
    answer = model_with_tool.invoke(messages)
print(answer.content)

# function call

In [ ]:
import os
from llms.langchain_tutorial.deepseek import get_deepseek_client

client = get_deepseek_client()

## define tools list

In [ ]:
from datetime import datetime

# 定义工具列表，模型在选择使用哪个工具时会参考工具的name和description
tools = [
    # 工具1 获取当前时刻的时间
    {
        "type": "function",
        "function": {
            "name": "get_current_time",
            "description": "当你想知道现在的时间时非常有用。",
            "parameters": {"type": "object", "properties": {}}
        }
    },  
    # 工具2 获取指定城市的天气
    {
        "type": "function",
        "function": {
            "name": "get_current_weather",
            "description": "当你想查询指定城市的天气时非常有用。",
            "parameters": {  # 查询天气时需要提供位置，因此参数设置为location
                        "type": "object",
                "properties": {
                    "location": {
                        "type": "string",
                        "description": "城市或县区，比如北京市、杭州市、余杭区等。"
                    }
                },
                "required": ["location"]
            }
        }
    }
]

# 模拟天气查询工具。返回结果示例：“北京今天是晴天。”
def get_current_weather(location):
    return f"{location}今天是雨天。 "

# 查询当前时间的工具。返回结果示例：“当前时间：2024-04-15 17:15:18。“
def get_current_time():
    # 获取当前日期和时间
    current_datetime = datetime.now()
    # 格式化当前日期和时间
    formatted_time = current_datetime.strftime('%Y-%m-%d %H:%M:%S')
    # 返回格式化后的当前时间
    return f"当前时间：{formatted_time}。"

## Invoke LLM with function calling

In [ ]:
import json

# DeepSeek 可以在同一轮请求多个工具；保留 tool_call_id 供下一轮使用。
def get_response(messages):
    return client.chat.completions.create(
        model=os.environ.get("DEEPSEEK_MODEL", "deepseek-flash"),
        messages=messages,
        tools=tools,
        extra_body={"thinking": {"type": "disabled"}},
    )


def call_with_messages():
    messages = [{"role": "user", "content": "杭州和北京天气怎么样？现在几点了？"}]
    functions_to_call = {
        "get_current_weather": get_current_weather,
        "get_current_time": get_current_time,
    }

    for round_no in range(1, 9):
        response = get_response(messages)
        assistant = response.choices[0].message
        print(f"第{round_no}轮大模型输出信息：{assistant}")
        if not assistant.tool_calls:
            print(f"最终答案：{assistant.content}")
            return assistant.content

        messages.append({
            "role": "assistant",
            "content": assistant.content or "",
            "tool_calls": [call.model_dump(exclude_none=True) for call in assistant.tool_calls],
        })
        for call in assistant.tool_calls:
            arguments = json.loads(call.function.arguments or "{}")
            if call.function.name == "get_current_weather":
                result = functions_to_call[call.function.name](arguments["location"])
            elif call.function.name == "get_current_time":
                result = functions_to_call[call.function.name]()
            else:
                raise ValueError(f"未知工具：{call.function.name}")
            print(f"工具输出信息：{result}")
            messages.append({
                "role": "tool",
                "tool_call_id": call.id,
                "content": result,
            })
    raise RuntimeError("工具调用超过 8 轮，请检查模型响应")


call_with_messages()

# LangChain Tools as OpenAI Functions
## Defining Custom Tools
### @tool decorator

In [ ]:
# Import things that are needed generically
from langchain.pydantic_v1 import BaseModel, Field
from langchain.tools import BaseTool, StructuredTool, tool

@tool
def get_current_weather(location: str) -> str:
    """当你想查询指定城市的天气时非常有用。"""
    return f"{location}今天是雨天。"

In [ ]:
get_current_weather.name, get_current_weather.description, get_current_weather.args

### tool decorator with JSON args

In [ ]:
class InputSchema(BaseModel):
    location: str = Field(description="城市或县区，比如北京市、杭州市、余杭区等。")

@tool("get_current_weather", args_schema=InputSchema)
def get_current_weather(location: str):
    """当你想查询指定城市的天气时非常有用。"""
    return f"{location}今天是雨天。"

In [ ]:
get_current_weather.name, get_current_weather.description, get_current_weather.args

### Subclass BaseTool

In [ ]:
from typing import Optional, Type

from langchain.callbacks.manager import (
    AsyncCallbackManagerForToolRun,
    CallbackManagerForToolRun,
)

class InputSchema(BaseModel):
    location: str = Field(description="城市或县区，比如北京市、杭州市、余杭区等。")

class GetCurrentWeatherTool(BaseTool):
    name = "get_current_weather"
    description = "当你想查询指定城市的天气时非常有用。"
    args_schema: Type[BaseModel] = InputSchema

    def _run(
        self, location: str, run_manager: Optional[CallbackManagerForToolRun] = None
    ) -> str:
        """Use the tool."""
        return f"{location}今天是雨天。"

In [ ]:
get_current_weather = GetCurrentWeatherTool()
get_current_weather.name, get_current_weather.description, get_current_weather.args

### StructuredTool dataclass

In [ ]:
class InputSchema(BaseModel):
    location: str = Field(description="城市或县区，比如北京市、杭州市、余杭区等。")

def get_current_weather_func(location: str):
    """当你想查询指定城市的天气时非常有用。"""
    return f"{location}今天是雨天。"

get_current_weather = StructuredTool.from_function(
    func=get_current_weather_func,
    name="get_current_weather",
    description="当你想查询指定城市的天气时非常有用。",
    args_schema=InputSchema
)

In [ ]:
get_current_weather.name, get_current_weather.description, get_current_weather.args

## Tools as OpenAI Functions 

In [ ]:
from langchain_core.messages import HumanMessage, AIMessage, ToolMessage
from llms.langchain_tutorial.tools import convert_to_openai_tool

In [ ]:
class WeatherSchema(BaseModel):
    location: str = Field(description="城市或县区，比如北京市、杭州市、余杭区等。")

@tool("get_current_weather", args_schema=WeatherSchema)
def get_current_weather(location: str):
    """当你想查询指定城市的天气时非常有用。"""
    return f"{location}今天是雨天。"


# 查询当前时间的工具。返回结果示例：“当前时间：2024-04-15 17:15:18。“
@tool("get_current_time")
def get_current_time():
    """当你想知道现在的时间时非常有用。"""
    # 获取当前日期和时间
    current_datetime = datetime.now()
    # 格式化当前日期和时间
    formatted_time = current_datetime.strftime('%Y-%m-%d %H:%M:%S')
    # 返回格式化后的当前时间
    return f"当前时间：{formatted_time}。"

functions = [get_current_weather, get_current_time]
tools = [convert_to_openai_tool(t) for t in functions]

In [ ]:
tools

In [ ]:
from llms.langchain_tutorial.deepseek import get_deepseek_chat

model = get_deepseek_chat()

messages = [HumanMessage(content="杭州和北京天气怎么样？现在几点了？")]

assistant_output = model.bind_tools(functions).invoke(messages)
assistant_output

In [ ]:
functions_to_call = {'get_current_weather': get_current_weather, 'get_current_time': get_current_time}

while assistant_output.tool_calls:
    messages.append(assistant_output)
    for tool in assistant_output.tool_calls:
        args = tool['args'] if 'properties' not in tool['args'] else tool['args']['properties']
        tool_content = functions_to_call[tool['name']].invoke(args)
        messages.append(ToolMessage(name=tool['name'], 
                                    tool_call_id=tool['id'], 
                                    content=tool_content)
                        )
        print(f"工具输出信息：{tool_content}\n")
        print("-"*60)
    
    assistant_output = model.bind_tools(functions).invoke(messages)
    print(f"大模型输出信息：{assistant_output}\n")

### bind_tools

In [ ]:
model_with_tools = model.bind_tools(functions)

messages = [HumanMessage(content="杭州和北京天气怎么样？现在几点了？")]

assistant_output = model_with_tools.invoke(messages)
assistant_output

# Agents
## Quickstart

In [ ]:
from langchain.prompts import (
    ChatPromptTemplate,
    PromptTemplate,
    HumanMessagePromptTemplate,
    MessagesPlaceholder,
)

prompt = ChatPromptTemplate.from_messages(
    [MessagesPlaceholder(variable_name='chat_history', optional=True),
     HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['input'], template='{input}')),
     MessagesPlaceholder(variable_name='agent_scratchpad')]
    )

In [ ]:

from langchain.pydantic_v1 import BaseModel, Field
from langchain.tools import tool

class WeatherSchema(BaseModel):
    location: str = Field(description="城市或县区，比如北京市、杭州市、余杭区等。")

@tool("get_current_weather", args_schema=WeatherSchema)
def get_current_weather(location: str):
    """当你想查询指定城市的天气时非常有用。"""
    return f"{location}今天是雨天。"


# 查询当前时间的工具。返回结果示例：“当前时间：2024-04-15 17:15:18。“
@tool("get_current_time")
def get_current_time():
    """当你想知道现在的时间时非常有用。"""
    # 获取当前日期和时间
    current_datetime = datetime.now()
    # 格式化当前日期和时间
    formatted_time = current_datetime.strftime('%Y-%m-%d %H:%M:%S')
    # 返回格式化后的当前时间
    return f"当前时间：{formatted_time}。"

tools = [get_current_weather, get_current_time]

In [ ]:
from langchain.agents import create_tool_calling_agent
from langchain.agents import AgentExecutor

from llms.langchain_tutorial.deepseek import get_deepseek_chat

model = get_deepseek_chat()

agent = create_tool_calling_agent(model, tools, prompt)
agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True)

In [ ]:
agent_executor.invoke({'input': '杭州和北京天气怎么样？现在几点了？'})

## XML Agent

In [ ]:
from langchain import hub
prompt = hub.pull("hwchase17/xml-agent-convo")
prompt.messages

In [ ]:
from datetime import datetime
from langchain.pydantic_v1 import BaseModel, Field

from llms.langchain_tutorial.tools import ModifyStructuredTool

class WeatherSchema(BaseModel):
    location: str = Field(description="城市或县区，比如北京市、杭州市、余杭区等。")

def get_current_weather_func(location: str):
    """当你想查询指定城市的天气时非常有用。"""
    return f"{location}今天是雨天。"


get_current_weather = ModifyStructuredTool.from_function(
    func=get_current_weather_func,
    name="get_current_weather",
    description="当你想查询指定城市的天气时非常有用。",
    args_schema=WeatherSchema
)


def get_current_time_func():
    """当你想知道现在的时间时非常有用。"""
    # 获取当前日期和时间
    current_datetime = datetime.now()
    # 格式化当前日期和时间
    formatted_time = current_datetime.strftime('%Y-%m-%d %H:%M:%S')
    # 返回格式化后的当前时间
    return f"当前时间：{formatted_time}。"


get_current_time = ModifyStructuredTool.from_function(
    func=get_current_time_func,
    name="get_current_time",
    description="当你想知道现在的时间时非常有用。",
)

tools = [get_current_weather, get_current_time]

In [ ]:
from langchain.agents import create_xml_agent
from langchain.agents import AgentExecutor

from llms.langchain_tutorial.deepseek import get_deepseek_chat

model = get_deepseek_chat()

agent = create_xml_agent(model, tools, prompt)
agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True, return_intermediate_steps=True)

In [ ]:
agent_executor.invoke({'input': '杭州和北京天气怎么样？现在几点了？'})

## Agent Iterator

In [ ]:
agent_executor = AgentExecutor(agent=agent, tools=tools)

for i, step in enumerate(agent_executor.iter({'input': '杭州和北京天气怎么样？现在几点了？'})):
    print(f'Agent的第 {i+1} 步输出:')
    if output := step.get("intermediate_step"):
        action, value = output[0]
        print(f'\taction: {action}')
        print(f'\tvalue: {value}')
    else:
        print(step['output'])

## Structured Output

In [ ]:
from typing import Dict

from langchain_core.agents import AgentActionMessageLog, AgentFinish, AgentAction
from langchain.agents.output_parsers.tools import parse_ai_message_to_tool_action
from langchain_core.pydantic_v1 import BaseModel, Field


class Response(BaseModel):
    """最终的回答如果包含天气、时间等信息，则需要调用该工具"""

    weather: Dict[str, str] = Field(
        description='最终的回答中不同城市对应的天气，比如北京的天气为阴天，则返回`{"北京": "阴天"}`。如果不存在天气信息则忽略', default={})
    current_time: str = Field(
        description="最终的回答中的当前时间，如果不存在则忽略", default="")


def parse(output):
    name = None
    # 判断调用的工具是否为 Response
    if output.tool_calls:
        tool_calls = output.tool_calls[0]
        name = tool_calls["name"]
        inputs = tool_calls["args"]
    elif output.additional_kwargs.get("tool_calls"):
        tool_calls = output.additional_kwargs["tool_calls"][0]
        name = tool_calls["name"]
        inputs = tool_calls["args"]
    
    if name == "Response":
        return AgentFinish(return_values=inputs, log=str(tool_calls))

    return parse_ai_message_to_tool_action(output)

In [ ]:
# prompt & tools的定义在上面[Agents-Quickstart]小节,不再重复编写
from langchain.agents.format_scratchpad import format_to_tool_messages
from langchain.agents import AgentExecutor
from llms.langchain_tutorial.deepseek import get_deepseek_chat

model = get_deepseek_chat()
llm_with_tools = model.bind_tools(tools + [Response])

agent = (
    {
        "input": lambda x: x["input"],
        # Format agent scratchpad from intermediate steps
        "agent_scratchpad": lambda x: format_to_tool_messages(
            x["intermediate_steps"]
        ),
    }
    | prompt
    | llm_with_tools
    | parse
)

In [ ]:
agent_executor = AgentExecutor(tools=tools, agent=agent, verbose=True)

agent_executor.invoke(
    {"input": "杭州和北京天气怎么样？现在几点了？"},
    return_only_outputs=True,
)

In [ ]:
from llms.langchain_tutorial.deepseek import get_deepseek_chat

print(get_deepseek_chat().invoke('请简要说明工具调用能帮助智能体完成什么任务。').content)